---
## Step 1: System Architecture

### RAG Pipeline Flow

![RAG pipleine](images/10_Building-a-Complete-RAG-Application.png)


### What Gets Tracked

**Experiment Level:**
- Model configuration
- Prompt versions
- Hyperparameters (top_k)

**Run Level:**
- Individual queries
- Performance metrics
- Cost per query

**Trace Level:**
- Every operation's timing
- Inputs and outputs
- Custom attributes

# Tutorial 1.9: Complete RAG Application

## Building a Production-Ready RAG System with Full Observability

Welcome to the penultimate notebook! This brings together everything you've learned to build a complete RAG (Retrieval-Augmented Generation) application with comprehensive MLflow tracking, tracing and evaluating.

### What You'll Build

A full RAG system that includes:
- ✅ Document embedding and indexing
- ✅ Semantic search / retrieval
- ✅ Context-aware response generation
- ✅ Complete experiment tracking
- ✅ End-to-end tracing
- ✅ Cost monitoring
- ✅ Performance metrics
- ✅ Caching strategies
- ✅ RAG evaluation with RAGAS metrics

### Prerequisites
- Completed all previous notebooks (1.1-1.8)
- Understanding of experiment tracking and tracing

### Estimated Time: 25-30 minutes

---
## Step 2: Environment Setup

In [ ]:
import time
import hashlib
from typing import List, Dict 
from utils.clnt_utils import is_databricks_ai_gateway_client, get_databricks_ai_gateway_client, get_ai_gateway_model_names, get_openai_client

import numpy as np
import mlflow
from dotenv import load_dotenv

# Load environment
load_dotenv()

EXPERIMENT_NAME = "09-complete-rag-system"
EMBEDDING_MODEL = "text-embedding-3-small"

# Configure MLflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

# Check if we are using a Databricks AI Gateway client
use_databricks_provider = is_databricks_ai_gateway_client()
print(f"   Use Databricks Provider: {use_databricks_provider}")

if use_databricks_provider:
    client = get_databricks_ai_gateway_client()
    models = get_ai_gateway_model_names()
    JUDGE_MODEL = models[2]
    AGENT_MODEL = models[0]
    JUDGE_MODEL_URI = f"databricks:/{JUDGE_MODEL}"
else:
    # Initialize and use as an OpenAI client
    client = get_openai_client()
    JUDGE_MODEL = "gpt-5.2"
    AGENT_MODEL = "gpt-5.2"
    JUDGE_MODEL_URI = f"openai:/{JUDGE_MODEL}"

# Enable autologging
mlflow.openai.autolog()

print("✅ Environment configured for production RAG system")
print(f"   MLflow tracking: {mlflow.get_tracking_uri()}")
print(f"   Experiment: {EXPERIMENT_NAME}")
print(f"   Model: {AGENT_MODEL}")
print(f"   Judge Model: {JUDGE_MODEL}")
print(f"   Use Databricks Provider: {use_databricks_provider}")

---
## Step 3: Document Store and Embeddings

In production, you'd use a vector database like Pinecone, Weaviate, Databricks Vector Store, or ChromaDB. For this tutorial, for simplicity and illustration, we'll use in-memory storage.

In [ ]:
# Sample document corpus (in production, load from database, comprising of text, .md, pdfs, etc.)
DOCUMENT_STORE = {
    "doc1": """MLflow is an open source developer platform to build AI applications and models with confidence.
It provides a unified interface for managing the entire machine learning lifecycle from experimentation to production deployment.
The platform supports both traditional ML workflows and modern GenAI application development under a single umbrella.
MLflow organizes work into experiments, runs, and traces, giving teams a consistent structure for tracking iterative development.
With a pluggable backend store and artifact repository, it scales from a local SQLite database to enterprise-grade cloud storage.""",

    "doc2": """MLflow Tracing provides comprehensive observability for GenAI applications built on the OpenTelemetry standard.
It captures LLM calls, retrieval steps, tool usage, and agent reasoning with full input/output visibility at every step.
Each trace is composed of typed spans such as LLM, RETRIEVER, TOOL, EMBEDDING, and CHAIN that reflect the logical structure of an application.
Traces can be searched, filtered, and analyzed in the MLflow UI to debug issues, identify latency bottlenecks, and optimize performance.
Because the tracing SDK is fully OpenTelemetry-compatible, teams avoid vendor lock-in and can export spans to any compatible backend.""",

    "doc3": """MLflow integrates with 40+ frameworks including OpenAI, Anthropic, LangChain, LlamaIndex, DSPy, and AutoGen.
Each integration provides automatic tracing and experiment tracking with minimal code changes, often just a single autolog call.
For example, calling mlflow.openai.autolog() captures every OpenAI SDK request as a traced span with token counts and latency.
LangChain and LlamaIndex integrations trace the full execution graph of chains, agents, and retrieval pipelines end to end.
These integrations ensure that teams can adopt MLflow incrementally without rewriting existing application code.""",

    "doc4": """MLflow Evaluation enables systematic testing of GenAI applications using LLM-as-judge metrics, custom scorers, and human feedback.
The mlflow.genai.evaluate() API accepts a dataset and a list of scorers, running each scorer against every example and aggregating the results.
Built-in scorers like Correctness, RelevanceToQuery, Safety, and Guidelines cover the most common quality dimensions out of the box.
Teams can also define custom scorers with the @scorer decorator to encode domain-specific quality criteria unique to their application.
It supports both batch evaluation against static datasets and online evaluation on live traces, making it easy to catch regressions before they reach production.""",

    "doc5": """MLflow Prompt Registry allows teams to version, share, and manage prompts centrally in a dedicated catalog.
Each prompt is stored as an immutable version, creating a full audit trail of how prompt text evolves over time.
Prompts can be linked to experiments and models, establishing traceability between the prompt used and the results observed.
The registry enables A/B testing of prompt variants so teams can iterate on prompt quality with data-driven decisions.
By decoupling prompt management from application code, non-engineering stakeholders can propose and review prompt changes without touching the codebase.""",

    "doc6": """MLflow supports collaborative development with experiment sharing, model versioning, and deployment tracking across teams.
Multiple team members can log runs to the same experiment, compare results side by side, and annotate findings with tags and notes.
The platform provides a centralized UI where data scientists and engineers can review each other's work without sharing notebooks or scripts.
Role-based access controls on Databricks allow organizations to set permissions at the experiment and model level for governance.
This shared workspace reduces duplication of effort and ensures that institutional knowledge is captured alongside the code and artifacts.""",

    "doc7": """MLflow provides cost tracking for LLM applications by monitoring token usage, API calls, and compute resources consumed per request.
When autologging is enabled, every LLM call automatically records prompt tokens, completion tokens, and total tokens as span attributes.
Teams can aggregate these token counts across traces to compute per-query, per-model, and per-experiment cost estimates.
The MLflow UI surfaces these metrics in experiment and trace views, giving teams real-time visibility into spending patterns.
This cost observability helps organizations optimize their LLM usage, choose the right model size for each task, and budget effectively at scale.""",

    "doc8": """MLflow is fully open source and vendor-neutral, ensuring no lock-in to any single cloud provider or LLM vendor.
It works with any cloud platform including AWS, Azure, and GCP, as well as on-premises infrastructure.
The platform supports every major ML framework and LLM provider, from PyTorch and scikit-learn to OpenAI, Anthropic, and Google.
A vibrant open source community contributes plugins, integrations, and extensions that continuously expand the ecosystem.
Organizations can self-host the MLflow tracking server or use managed offerings like Databricks without changing their application code.""",

    "doc9": """MLflow is a platform for the complete machine learning lifecycle, covering experimentation, reproducibility, and governance.
It provides experiment tracking that logs parameters, metrics, and artifacts for every run, enabling systematic comparison of approaches.
Model packaging with the MLmodel format ensures that models are portable across serving environments regardless of the training framework.
The platform captures environment dependencies automatically, so any run can be reproduced months or years later with identical results.
MLflow's governance features include model staging workflows, approval gates, and lineage tracking from data to deployed endpoint.""",

    "doc10": """MLflow offers a suite of monitoring and quality tools including performance metrics dashboards, a Judge Builder for custom evaluators, and continuous online monitoring.
The Judge Builder lets teams define LLM-as-judge evaluators through a guided interface without writing scorer code from scratch.
Online monitoring automatically evaluates incoming traces against registered scorers, surfacing quality regressions as they happen in production.
Performance dashboards aggregate latency, error rates, and token usage across experiments, giving teams a real-time operational overview.
Together these tools give teams end-to-end insight into application health, from development-time evaluation to production-time quality assurance.""",

    "doc11": """MLflow's Model Registry provides a centralized model store for managing the full lifecycle of ML models from development to production.
Each registered model maintains a version history, so teams can roll back to any previous version if a new deployment introduces regressions.
The registry supports annotations and descriptions on every model version, capturing context about training data, performance benchmarks, and intended use.
Approval workflows and stage transitions (such as Staging to Production) enforce governance policies before a model is promoted to serve live traffic.
Integration with CI/CD pipelines enables automated model validation and promotion, streamlining the path from experiment to deployment.""",

    "doc12": """MLflow Deployments enable serving models as REST API endpoints with built-in monitoring, authentication, and autoscaling.
Teams can deploy models to various targets including local servers, cloud platforms, and Kubernetes clusters using a single unified command.
Each deployment is linked back to its registered model version, maintaining full lineage from training run to production endpoint.
The deployment framework supports A/B testing and canary rollouts, allowing teams to gradually shift traffic between model versions.
Built-in health checks and request logging ensure that deployed models are observable and that any serving issues are detected quickly.""",
}
print(f"📚 Document store initialized with {len(DOCUMENT_STORE)} documents")

### Create embedding for a query

In [ ]:
# Memory Cache for embeddings but in production, use Redis or something similar vector database
# that offer caching mechanism. If on Databricks, you can use the Databricks Vectore Store.

EMBEDDING_CACHE = {}

# creating a span for the embedding function, allowing us
# trace the embedding function and see the time it takes to embed the text
# and the number of tokens it uses.
@mlflow.trace(name="embed_text", span_type="EMBEDDING")
def embed_text(text: str) -> List[float]:
    """
    Generate embeddings with caching.
    """
    # Check cache
    cache_key = hashlib.md5(text.encode()).hexdigest()
    # get the current span and set the attributes for the span
    span = mlflow.get_current_active_span()
    # if the cache key is in the cache, set the cache hit to true and 
    # return the cached embedding
    if cache_key in EMBEDDING_CACHE:
        span.set_attributes({"cache_hit": True})
        return EMBEDDING_CACHE[cache_key]

    span.set_attributes({"cache_hit": False, "text_length": len(text)})
    
    # Generate embedding using the OpenAI embedding model
    response = client.embeddings.create(
        model= EMBEDDING_MODEL,
        input=text
    )
    
    embedding = response.data[0].embedding
    span.set_attributes({"embedding_dim": len(embedding)})
    
    # Cache for future use
    EMBEDDING_CACHE[cache_key] = embedding
    
    return embedding

print("✅ Embedding function defined with caching")

### Compute embedings for each doc and add them to the memory store.

In [ ]:
# Pre-compute document embeddings
print("\n🔄 Computing document embeddings...")

# Our in-memory store for the document embeddings
DOC_EMBEDDINGS = {}

for doc_id, text in DOCUMENT_STORE.items():
    DOC_EMBEDDINGS[doc_id] = embed_text(text)
    print(f"  ✓ {doc_id}")

print(f"\n✅ {len(DOC_EMBEDDINGS)} documents embedded and cached")

---
## Step 4: Query Processing and Validation

1. check for minimal or maximum length
2. check for offensive content
3. check any custom validation

In [ ]:
@mlflow.trace(name="validate_query", span_type="PARSER")
def validate_query(query: str) -> Dict:
    """
    Validate and preprocess user query.
    """
    span = mlflow.get_current_active_span()
    span.set_attributes({"original_length": len(query)})
    
    # Basic validation
    if not query or len(query.strip()) == 0:
        mlflow.log_span_attribute("validation_error", "empty_query")
        raise ValueError("Query cannot be empty")
    
    if len(query) > 1000:
        span.set_attributes({"validation_warning": "query_too_long"})
        query = query[:1000]  # Truncate
    
    # Preprocess
    processed_query = query.strip()
    span.set_attributes({
        "processed_length": len(processed_query),
        "validation_passed": True
    })
    
    return {
        "original": query,
        "processed": processed_query,
        "valid": True
    }

print("✅ Query validation function defined")

---
## Step 5: Semantic Search with Similarity Scoring

In [ ]:
@mlflow.trace(name="semantic_search", span_type="RETRIEVER")
def search_documents(
    query_embedding: List[float],
    doc_embeddings: Dict[str, List[float]],
    top_k: int = 3,
    min_score: float = 0.7
) -> List[Dict]:
    """
    Search for most relevant documents using cosine similarity.
    """
    span = mlflow.get_current_active_span()

    # set the attributes for the span
    span.set_attributes({
        "corpus_size": len(doc_embeddings),
        "top_k": top_k,
        "min_score": min_score
    })
    
    # Score every document by cosine similarity to the query embedding.
    # Cosine similarity = (a · b) / (||a|| * ||b||), the cosine of the angle
    # between two vectors. Bounded in [-1, 1]; ~1 = same direction (semantically
    # similar), ~0 = orthogonal (unrelated), ~-1 = opposite. We use it instead of
    # raw dot product so vector magnitude (e.g. document length) doesn't bias the
    # score — only the *direction* of the embedding matters.
    scores = {}
    for doc_id, doc_emb in doc_embeddings.items():
        # Numerator: dot product — measures directional alignment of the two embeddings.
        # Denominator: product of L2 norms — normalizes both vectors to unit length.
        similarity = np.dot(query_embedding, doc_emb) / \
                     (np.linalg.norm(query_embedding) * np.linalg.norm(doc_emb))
        scores[doc_id] = float(similarity)
    
    # Sort by score and filter by minimum threshold
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    filtered_docs = [(doc_id, score) for doc_id, score in sorted_docs if score >= min_score]
    
    # Get top k
    top_docs = filtered_docs[:top_k]
    
    # Return in LangChain Document format (page_content + metadata)
    # MLflow's RAGAS integration expects this format in RETRIEVER span outputs
    results = [
        {
            "page_content": DOCUMENT_STORE[doc_id],
            "metadata": {"doc_id": doc_id, "score": score}
        }
        for doc_id, score in top_docs
    ]
    
    # Set span metrics
    if results:
        span.set_attributes({
            "num_results": len(results),
            "top_score": results[0]["metadata"]["score"],
            "avg_score": np.mean([r["metadata"]["score"] for r in results]),
            "min_result_score": results[-1]["metadata"]["score"]
        })
    else:
        span.set_attributes({"num_results": 0,
                             "retrieval_warning": "no_docs_above_threshold"})
    
    return results

print("✅ Semantic search function defined")

---
## Step 6: Context Assembly and Prompt Construction

In [ ]:
@mlflow.trace(name="assemble_context", span_type="PARSER")
def assemble_context(query: str, retrieved_docs: List[Dict]) -> str:
    """
    Assemble context from retrieved documents and construct prompt.
    """
    span = mlflow.get_current_active_span()

    # set the attributes for the span
    span.set_attributes({"num_docs": len(retrieved_docs)})
    
    if not retrieved_docs:
        span.set_attributes({"context_warning": "no_docs_retrieved"})
        return None
    
    # Format context
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        context_parts.append(
            f"[Document {i}] (Relevance: {doc['metadata']['score']:.2f})\n{doc['page_content']}"
        )
    
    context = "\n\n".join(context_parts)
    
    # Construct prompt with system instructions
    prompt = f"""You are a helpful AI assistant that answers questions based on provided context.

INSTRUCTIONS:
- Answer the question using ONLY the information in the context below
- If the answer is not in the context, say "I don't have enough information to answer that"
- Be concise but complete
- Cite which document(s) you used if relevant

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""
    span.set_attributes({
        "context_length": len(context),
        "prompt_length": len(prompt)
    })
    
    return prompt

print("✅ Context assembly function defined")

---
## Step 7: LLM Response Generation

In [ ]:
@mlflow.trace(name="generate_response", span_type="LLM")
def generate_answer(
    prompt: str,
    model: str = "gpt-5.2",
    temperature: float = 0.1,
) -> Dict:
    """
    Generate answer using LLM.
    """
    span = mlflow.get_current_active_span()

    # set the attributes for the span
    span.set_attributes({
        "model": model,
        "temperature": temperature
    })
    
    if not prompt:
        span.set_attributes({"generation_error": "empty_prompt"})
        raise ValueError("Prompt cannot be empty")
    
    # Call LLM (automatically traced by OpenAI autolog)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    
    answer = response.choices[0].message.content
    
    # Set generation attributes for the span
    span.set_attributes({
        "answer_length": len(answer),
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
        "finish_reason": response.choices[0].finish_reason
    })
    
    return {
        "answer": answer,
        "tokens": response.usage.total_tokens,
        "model": model,
        "finish_reason": response.choices[0].finish_reason
    }

print("✅ Answer generation function defined")

---
## Step 8: Response Validation and Post-Processing

Like preprocessing, this could be basic validation and custom post-processing.

In [ ]:
@mlflow.trace(name="validate_response", span_type="PARSER")
def validate_response(answer: str, min_length: int = 10) -> Dict:
    """
    Validate and post-process generated response.
    """
    span = mlflow.get_current_active_span()
    span.set_attributes({
        "min_length_threshold": min_length,
        "answer_length": len(answer)
    })
    
    # Keep track of issues
    issues = []
    
    # Check length
    if len(answer) < min_length:
        issues.append("too_short")
    
    # Check for common error patterns
    error_patterns = [
        "I don't have enough information",
        "I cannot answer",
        "I apologize"

    ]
    
    for pattern in error_patterns:
        if pattern.lower() in answer.lower():
            issues.append(f"contains_{pattern.replace(' ', '_').lower()}")
    
    span.set_attributes({
        "validation_issues": ",".join(issues) if issues else "none",
        "is_valid": len(issues) == 0
    })
    
    return {
        "is_valid": len(issues) == 0,
        "issues": issues,
        "answer": answer
    }

print("✅ Response validation function defined")

---
## Step 9: Complete RAG Pipeline

All the trace functions defiend above, with `span_type` attribute come to bear here as part of the pipeline

In [ ]:
@mlflow.trace(name="rag_pipeline", span_type="CHAIN")
def rag_qa_system(
    user_query: str,
    top_k: int = 3,             # return top k documents
    min_score: float = 0.6,     # minimum score for a document to be retrieved
    model: str = "gpt-5.2",     # model to use for generation
    temperature: float = 0.1    # temperature for generation
) -> Dict:
    """
    Complete RAG pipeline with full observability.
    
    Returns:
        Dict with answer, metadata, and status
    """
    try:
        # Execute the pipeline in series of sequential steps as
        # functions defined above, with respective span_types

        # Step 1: Validate query
        validation_result = validate_query(user_query)

        # Extract the processed query from the validation result
        processed_query = validation_result["processed"]
        
        # Step 2: Generate query embedding
        query_embedding = embed_text(processed_query)
        
        # Step 3: Search documents
        retrieved_docs = search_documents(
            query_embedding,
            DOC_EMBEDDINGS,
            top_k=top_k,
            min_score=min_score
        )
        
        if not retrieved_docs:
            return {
                "status": "no_relevant_docs",
                "answer": "I couldn't find relevant information to answer your question.",
                "query": user_query,
                "retrieved_docs": []
            }
        
        # Step 4: Assemble context and construct prompt
        prompt = assemble_context(processed_query, retrieved_docs)
        
        # Step 5: Generate answer by calling the LLM
        generation_result = generate_answer(
            prompt,
            model=model,
            temperature=temperature
        )
        
        # Step 6: Validate response
        validation = validate_response(generation_result["answer"])
        
        # Construct result
        result = {
            "status": "success",
            "query": user_query,
            "answer": generation_result["answer"],
            "retrieved_docs": retrieved_docs,
            "metadata": {
                "num_docs": len(retrieved_docs),
                "avg_relevance": float(np.mean([d["metadata"]["score"] for d in retrieved_docs])),
                "tokens_used": generation_result["tokens"],
                "model": model,
                "is_valid": validation["is_valid"],
                "validation_issues": validation["issues"]
            }
        }
        
        return result
        
    except Exception as e:
        return {
            "status": "error",
            "query": user_query,
            "error": str(e),
            "error_type": type(e).__name__
        }

print("✅ Complete RAG pipeline defined")

---
## Step 10: Testing the RAG System

In [ ]:
# Test queries
test_queries = [
    "What tracing capabilities does MLflow provide?",
    "How does MLflow help with cost tracking?",
    "Can MLflow integrate with LangChain?",
    "What is the purpose of MLflow Prompt Registry?",
    "How does the MLflow Model Registry manage model versions and deployments?",
]

print("\n🧪 Testing RAG System\n")
print("="*80)

results = []

for i, query in enumerate(test_queries, 1):
    print(f"\nQuery {i}: {query}")
    print("-"*80)
    
    start_time = time.time()
    # Execute the pipeline for each query
    result = rag_qa_system(query, top_k=3, min_score=0.6, model=AGENT_MODEL)
    latency = time.time() - start_time
    
    if result["status"] == "success":
        print(f"\nAnswer: {result['answer']}")
        doc_ids = [d["metadata"]["doc_id"] for d in result["retrieved_docs"]]
        print("\nMetadata:")
        print(f"  - Documents used: {result['metadata']['num_docs']} → {', '.join(doc_ids)}")
        print(f"  - Avg relevance: {result['metadata']['avg_relevance']:.3f}")
        print(f"  - Tokens: {result['metadata']['tokens_used']}")
        print(f"  - Latency: {latency:.2f}s")
        print(f"  - Valid: {result['metadata']['is_valid']}")
        
        results.append({
            "query": query,
            "success": True,
            "latency": latency,
            "tokens": result['metadata']['tokens_used']
        })
    elif result["status"] == "no_relevant_docs":
        print("\n⚠️  No relevant documents found (all below min_score threshold)")
        print(f"  - Latency: {latency:.2f}s")
        results.append({
            "query": query,
            "success": False,
            "latency": latency
        })
    else:
        print(f"\n❌ Error ({result.get('error_type', 'unknown')}): {result.get('error', 'No details')}")
        results.append({
            "query": query,
            "success": False,
            "latency": latency
        })

print("\n" + "="*80)
print("\n✅ All queries processed!")

---
## Step 11: Performance Analysis

Before running formal evaluation with RAGAS scorers (Step 12), we first examine the **operational health** of the RAG pipeline using the results collected during testing. This analysis covers three dimensions:

| Dimension | What It Tells You |
|-----------|-------------------|
| **Latency** | How fast is the end-to-end pipeline? Are there outliers that signal retrieval or LLM bottlenecks? |
| **Token usage** | How many tokens does each query consume? What's the estimated cost per query and in aggregate? |
| **Cache effectiveness** | How many embeddings are cached? Re-runs should show near-100% cache hits, reducing both latency and cost. |

This is a quick sanity check — if latency is too high or token counts are unexpectedly large, you'd want to investigate before investing in a full quality evaluation.

In [ ]:
# Analyze performance
print("\n📊 Performance Summary\n")
print("="*60)

successful = [r for r in results if r["success"]]

if successful:
    latencies = [r["latency"] for r in successful]
    tokens = [r["tokens"] for r in successful]
    
    print(f"Success Rate: {len(successful)}/{len(results)} ({len(successful)/len(results)*100:.1f}%)")
    print("\nLatency Stats:")
    print(f"  Average: {np.mean(latencies):.2f}s")
    print(f"  Min: {np.min(latencies):.2f}s")
    print(f"  Max: {np.max(latencies):.2f}s")
    print(f"  Std Dev: {np.std(latencies):.2f}s")
    
    print("\nToken Usage:")
    print(f"  Average: {np.mean(tokens):.0f} tokens")
    print(f"  Total: {np.sum(tokens):.0f} tokens")
    print(f"  Est. Cost: ${np.sum(tokens) * 0.15 / 1_000_000:.6f}")
    
    print("\nCache Performance:")
    print(f"  Embedding cache hits: {len(EMBEDDING_CACHE)} embeddings cached")
    
print("\n" + "="*60)

---
## Step 12: RAG Evaluation with RAGAS Metrics

Now let's evaluate our RAG system using [RAGAS](https://www.ragas.io/) (Retrieval-Augmented Generation Assessment) metrics. These metrics help assess the quality of both retrieval and generation.

| Metric | What It Measures | Requirements |
|--------|-----------------|--------------|
| **Faithfulness** | Does the answer only use facts from the retrieved documents, without adding or inventing information? | Traces with RETRIEVER spans |
| **ContextRelevance** | Did the retriever fetch documents that actually help answer the question, rather than unrelated content? | Traces with RETRIEVER spans |

**Important:** RAGAS scorers extract context from **traces with RETRIEVER spans**, not from static datasets. This is why we search for traces from our RAG pipeline rather than constructing a manual dataset.

In [ ]:
# Workaround for async event loop issues in Jupyter notebooks
# Libraries like RAGAS run async code that can conflict with Jupyter's event loop

import logging

# 1. Use nest_asyncio to allow nested event loops (essential for Jupyter + async libraries)
import nest_asyncio
nest_asyncio.apply()

# 2. Suppress noisy asyncio error messages (they don't affect results)
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

# 3. If litellm is installed, disable its async logging to prevent conflicts
try:
    import litellm
    litellm.success_callback = []
    litellm.failure_callback = []
    litellm._async_success_callback = []
    litellm._async_failure_callback = []
    litellm.disable_streaming_logging = True
    litellm.turn_off_message_logging = True
    logging.getLogger("LiteLLM").setLevel(logging.WARNING)
    print("   - LiteLLM async logging disabled")
except ImportError:
    pass  # litellm not installed, no workaround needed

print("✅ Jupyter async compatibility configured")
print("   - nest_asyncio applied for nested event loop support")
print("   - Asyncio error logging suppressed")

In [ ]:
from mlflow.genai.scorers.ragas import Faithfulness, ContextRelevance


# Initialize RAGAS scorers (requires: pip install ragas)
# Note: We're using Faithfulness and ContextRelevance which work with traces containing RETRIEVER spans
# ContextPrecision is not included because it requires expectations['expected_output']

faithfulness_scorer = Faithfulness(model=JUDGE_MODEL_URI)
context_relevance_scorer = ContextRelevance(model=JUDGE_MODEL_URI)

print("✅ RAGAS scorers initialized:")
print("   - Faithfulness (checks if answer is grounded in retrieved context)")
print("   - ContextRelevance (checks if retrieved context is relevant to query)")

In [ ]:
from mlflow.entities import Trace as TraceEntity

def _has_valid_retriever_output(trace_obj):
    """Return True if the trace has a RETRIEVER span with non-empty page_content outputs."""
    if isinstance(trace_obj, str):
        trace_obj = TraceEntity.from_json(trace_obj)
    for span in trace_obj.data.spans:
        if span.span_type == "RETRIEVER":
            outputs = span.outputs
            if (isinstance(outputs, list) and len(outputs) > 0
                    and isinstance(outputs[0], dict) and "page_content" in outputs[0]):
                return True
    return False

# Get traces from the RAG pipeline runs (which contain RETRIEVER spans)
# RAGAS scorers like Faithfulness and ContextRelevance extract context
# from RETRIEVER spans in traces, not from static datasets

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# Fetch extra traces as buffer — some may have empty retrieval results
# (e.g. queries where no document exceeded min_score) which would cause
# RAGAS scorers to fail with "missing retrieval spans"
rag_traces = mlflow.search_traces(
    locations=[experiment.experiment_id],
    filter_string="name = 'rag_pipeline'",
    max_results=len(test_queries) * 2,
)

# Keep only traces whose RETRIEVER spans returned documents with page_content
total_before = len(rag_traces)
valid_mask = rag_traces["trace"].apply(_has_valid_retriever_output)
rag_traces = rag_traces[valid_mask].head(len(test_queries)).reset_index(drop=True)
excluded = total_before - valid_mask.sum()

print(f"✅ Found {total_before} RAG pipeline traces, kept {len(rag_traces)} with valid RETRIEVER outputs")
if excluded:
    print(f"   ⚠️  Excluded {excluded} trace(s) with empty retrieval results (RAGAS requires non-empty RETRIEVER outputs)")

print("\nThese traces contain RETRIEVER spans that RAGAS scorers need:")
print("   - semantic_search span with retrieved documents")
print("   - Full input/output flow for faithfulness checking")

# Verify sample trace structure
sample_trace = rag_traces.iloc[0]["trace"]
if isinstance(sample_trace, str):
    sample_trace = TraceEntity.from_json(sample_trace)
retriever_spans = [s for s in sample_trace.data.spans if s.span_type == "RETRIEVER"]
if retriever_spans:
    outputs = retriever_spans[0].outputs
    if isinstance(outputs, list) and len(outputs) > 0 and isinstance(outputs[0], dict):
        print(f"\n   Retriever output keys: {list(outputs[0].keys())}")
        print(f"   Has page_content: {'page_content' in outputs[0]}")

### Run evaluation on the traces

In [ ]:
# Run RAGAS evaluation on traces (which contain RETRIEVER spans)

print("🔄 Running RAGAS evaluation on RAG traces...\n")

ragas_results = mlflow.genai.evaluate(
    data=rag_traces,  # Pass traces (not static data) - they contain RETRIEVER spans
    scorers=[
        faithfulness_scorer,      # Checks if answer is grounded in retrieved context
        context_relevance_scorer, # Checks if retrieved context is relevant to query
    ]
)

print("\n✅ RAGAS evaluation complete!")
print("\n📊 RAGAS Metrics Summary:")
print("-" * 50)
for metric_name, value in ragas_results.metrics.items():
    if isinstance(value, float):
        print(f"  {metric_name}: {value:.3f}")
    else:
        print(f"  {metric_name}: {value}")

---
## Step 13: Viewing Results in MLflow UI

Now let's explore what was tracked in MLflow.

### Analyzing Results in MLflow UI

**Experiments View** — Navigate to http://localhost:5000 and select the experiment.

You'll see:
- All RAG query traces
- Span attributes (model, top_k, relevance scores)
- Timing for each pipeline step

**Traces View** — Click the "Traces" tab to see the full pipeline timeline:

```
rag_pipeline (CHAIN) ━━━━━━━━━━━━━━━━━━━━ 2.5s
├─ validate_query (PARSER) ━━ 0.01s
├─ embed_text (EMBEDDING) ━━━━ 0.3s
├─ semantic_search (RETRIEVER) ━ 0.05s
├─ assemble_context (PARSER) ━ 0.02s
├─ generate_response (LLM) ━━━━━ 2.0s
│  └─ OpenAI API call ━━━━━━━━━ 1.9s
└─ validate_response (PARSER) ━ 0.01s
```

**Span Details** — Click on any span to see:
- Inputs and outputs
- Custom attributes (cache hit status, relevance scores, token counts)
- Timing information

**Key Insights from Traces:**

| Category | What to Look For |
|----------|-----------------|
| **Performance Bottlenecks** | Which step takes longest? Is it the LLM or retrieval? |
| **Quality Metrics** | Average relevance scores, documents per query, answer validation rates |
| **Cost Analysis** | Token usage per query, cache effectiveness, cost per operation |
| **Error Patterns** | Failed queries, low relevance scores, validation issues |

**Optimization Opportunities** — Based on traces, you can:
- Adjust `top_k` if retrieval is slow
- Increase `min_score` if quality is poor
- Optimize prompts to reduce tokens
- Add more aggressive caching
- Implement parallel retrieval

---
## Step 14: Summary

In this notebook, you learned how to build a **production-ready RAG application** with full MLflow observability:

1. Building an **end-to-end RAG pipeline**: embedding, semantic search, context assembly, generation, and validation
2. Computing **cosine similarity** retrieval over an in-memory document store (stand-in for a production vector DB)
3. Adding **typed spans** (`EMBEDDING`, `RETRIEVER`, `LLM`, `PARSER`, `CHAIN`) with `@mlflow.trace(span_type=...)` so traces are structured, not opaque
4. Decorating spans with **custom attributes** (`top_k`, `min_score`, `top_score`, `cache_hit`, token counts) for downstream analysis
5. Tracking **cost, latency, and retrieval quality** per query using span attributes and run metrics
6. Evaluating RAG with **RAGAS scorers** (`Faithfulness`, `ContextRelevance`) that read context from `RETRIEVER` spans in traces — not from static datasets
7. Implementing a **caching layer** for embeddings and inspecting cache effectiveness through traces

### Key Takeaways

- **Span types are not cosmetic**: `RETRIEVER` spans drive RAGAS evaluation, `LLM` spans drive cost/token tracking, `EMBEDDING` spans isolate vector-generation latency. Tagging them correctly unlocks downstream tooling for free.
- **RAGAS reads traces, not datasets**: `Faithfulness` and `ContextRelevance` extract retrieved context from the trace itself. Your `predict_fn` must produce a real trace with a `RETRIEVER` span containing `page_content` for the scorer to work.
- **LangChain Document format matters**: RAGAS expects retrieved docs as `{"page_content": ..., "metadata": {...}}`. Returning bare strings or custom dicts breaks the integration silently.
- **Cosine similarity over raw dot product**: Normalizing by `||a|| * ||b||` removes magnitude bias (e.g. document length) so only semantic direction matters.
- **Trace-first design wins**: When every pipeline step is a span with attributes, debugging, cost analysis, and quality eval all share one source of truth — no separate logging pipeline needed.
- **In-memory is fine for tutorials; not for production**: Swap `DOCUMENT_STORE` for Pinecone / Weaviate / ChromaDB / Databricks Vector Search before shipping.

### What's Next?

**Notebook 1.10: Multi-Agent Supervisor with LangGraph**

Learn how to:
- Implement the **supervisor pattern** with LangGraph `StateGraph`
- Route between specialist agents (Genie text-to-SQL + Knowledge Assistant)
- Auto-trace multi-agent execution with `mlflow.langchain.autolog()`
- Evaluate agents with built-in scorers, custom routing accuracy, and **Agent-as-a-Judge**

### Additional Resources

- [MLflow Tracing](https://mlflow.org/docs/latest/genai/tracing/index.html)
- [MLflow GenAI Evaluation](https://mlflow.org/docs/latest/genai/evaluation/index.html)
- [RAGAS Documentation](https://docs.ragas.io/)
- [Databricks Vector Search](https://docs.databricks.com/en/generative-ai/vector-search.html)